# Resonant Sparse Geometry Networks (RSGN)
# Full Manuscript Experiments

This notebook runs all experiments for the RSGN manuscript with genuine results.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**

**Experiments:**
1. Hierarchical Classification (20 classes, 6 models, 3 runs each)
2. Long-Range Dependencies (sequence length 128, 4 models, 3 runs each)
3. Ablation Study (6 configurations)

**Output:** JSON results, PDF figures, LaTeX tables

In [ ]:
#@title Cell 1: Environment Setup {display-mode: "form"}
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os
import time
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict, deque
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"="*60)
print(f"RSGN Manuscript Experiments")
print(f"="*60)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

# Create output directories
os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)

def set_seed(seed=42):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"\nSetup complete!")
print(f"="*60)

In [ ]:
#@title Cell 2: Hyperbolic Geometry Operations {display-mode: "form"}

def poincare_distance(x, y, eps=1e-5):
    """Compute hyperbolic distance in Poincare ball model."""
    diff_norm_sq = torch.sum((x - y) ** 2, dim=-1)
    x_norm_sq = torch.sum(x ** 2, dim=-1)
    y_norm_sq = torch.sum(y ** 2, dim=-1)
    numerator = 2 * diff_norm_sq
    denominator = (1 - x_norm_sq) * (1 - y_norm_sq) + eps
    arg = 1 + numerator / denominator
    return torch.acosh(torch.clamp(arg, min=1 + eps))


def project_to_poincare(x, max_norm=0.99, eps=1e-5):
    """Project points to Poincare ball interior."""
    norm = torch.norm(x, dim=-1, keepdim=True)
    scale = torch.where(norm >= 1, max_norm / (norm + eps), torch.ones_like(norm))
    return x * scale


class HyperbolicDistance(nn.Module):
    """Batched hyperbolic distance computation."""
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        # x: [batch, N, dim], y: [batch, M, dim]
        x_exp = x.unsqueeze(2)  # [batch, N, 1, dim]
        y_exp = y.unsqueeze(1)  # [batch, 1, M, dim]
        diff_norm_sq = torch.sum((x_exp - y_exp) ** 2, dim=-1)
        x_norm_sq = torch.sum(x_exp ** 2, dim=-1)
        y_norm_sq = torch.sum(y_exp ** 2, dim=-1)
        numerator = 2 * diff_norm_sq
        denominator = (1 - x_norm_sq) * (1 - y_norm_sq) + self.eps
        arg = 1 + numerator / denominator
        return torch.acosh(torch.clamp(arg, min=1 + self.eps))


print("Hyperbolic geometry operations defined.")

In [ ]:
#@title Cell 3: RSGN Model Implementation {display-mode: "form"}

class RSGNetwork(nn.Module):
    """Resonant Sparse Geometry Network.
    
    A brain-inspired neural network that combines:
    - Hyperbolic geometry for hierarchical representations
    - Sparse activation through input-dependent ignition
    - Distance-based connectivity in Poincare ball
    - Two-timescale learning (fast gradient + slow Hebbian)
    """
    
    def __init__(self, num_nodes=256, dim_input=64, dim_hidden=128, dim_space=3,
                 num_steps=5, temperature=1.0, tau=1.0, sparsity_target=0.1,
                 inhibition_radius=0.3):
        super().__init__()
        self.num_nodes = num_nodes
        self.dim_input = dim_input
        self.dim_hidden = dim_hidden
        self.dim_space = dim_space
        self.num_steps = num_steps
        self.temperature = temperature
        self.tau = tau
        self.sparsity_target = sparsity_target
        self.inhibition_radius = inhibition_radius

        # Node positions in Poincare ball (learnable)
        self.positions = nn.Parameter(torch.randn(num_nodes, dim_space) * 0.3)
        
        # Per-node activation thresholds
        self.thresholds = nn.Parameter(torch.ones(num_nodes) * 0.5)
        
        # Hierarchical levels
        self.levels = nn.Parameter(torch.linspace(0, 1, num_nodes))

        # Factorized affinity matrix (low-rank approximation)
        affinity_rank = 32
        self.affinity_u = nn.Parameter(torch.randn(num_nodes, affinity_rank) * 0.01)
        self.affinity_v = nn.Parameter(torch.randn(num_nodes, affinity_rank) * 0.01)

        # Input embedding to hyperbolic space
        self.input_embed = nn.Sequential(
            nn.Linear(dim_input, dim_space * 4),
            nn.GELU(),
            nn.Linear(dim_space * 4, dim_space),
            nn.Tanh()
        )
        self.input_scale = nn.Parameter(torch.tensor(0.7))

        # State transformation
        self.W_h = nn.Linear(dim_hidden, dim_hidden)
        self.layer_norm = nn.LayerNorm(dim_hidden)
        self.init_state = nn.Linear(dim_input, dim_hidden)
        
        # Hyperbolic distance module
        self.hyp_dist = HyperbolicDistance()

    def get_positions(self):
        """Get node positions projected to Poincare ball."""
        return project_to_poincare(self.positions, max_norm=0.95)

    def compute_connection_weights(self):
        """Compute connection weights based on hyperbolic distance and affinity."""
        pos = self.get_positions()
        pos_batch = pos.unsqueeze(0)  # [1, N, dim]
        
        # Hyperbolic distance matrix
        dist = self.hyp_dist(pos_batch, pos_batch)[0]  # [N, N]
        
        # Distance-based weights (exponential decay)
        dist_weight = torch.exp(-dist / self.tau)
        
        # Learned affinity (low-rank factorization)
        affinity = torch.sigmoid(self.affinity_u @ self.affinity_v.T)
        
        # Hierarchical level factor (prefer feedforward flow)
        level_diff = self.levels.unsqueeze(1) - self.levels.unsqueeze(0)
        level_factor = F.softplus(level_diff + 1)
        
        # Combined weight matrix
        W = affinity * dist_weight * level_factor
        
        # Remove self-connections
        W = W * (1 - torch.eye(self.num_nodes, device=W.device))
        
        return W

    def soft_threshold(self, x, theta):
        """Soft threshold activation."""
        return torch.sigmoid((x - theta) / self.temperature)

    def ignite(self, x):
        """Input-dependent ignition: map inputs to node activations."""
        batch_size, seq_len, _ = x.shape
        
        # Embed inputs to hyperbolic space
        sparks = self.input_embed(x) * self.input_scale
        sparks = project_to_poincare(sparks, max_norm=0.9)
        
        # Get node positions
        pos = self.get_positions()
        
        # Compute distance from each input to each node
        sparks_exp = sparks.unsqueeze(2)  # [batch, seq, 1, dim]
        pos_exp = pos.unsqueeze(0).unsqueeze(0)  # [1, 1, N, dim]
        dist_sq = ((sparks_exp - pos_exp) ** 2).sum(-1)  # [batch, seq, N]
        
        # Find minimum distance across sequence
        min_dist_sq = dist_sq.min(dim=1)[0]  # [batch, N]
        
        # Convert distance to activation
        sigma_ign = 0.4
        activation = torch.exp(-min_dist_sq / (2 * sigma_ign ** 2))
        
        return activation

    def propagate(self, h, alpha, W):
        """Propagate signals through the network."""
        # Mask inactive nodes
        active_mask = (alpha > 0.01).float()
        h_weighted = h * alpha.unsqueeze(-1) * active_mask.unsqueeze(-1)
        
        # Message passing
        h_msg = torch.einsum('ij,bjd->bid', W.T, h_weighted)
        h_new = self.W_h(h_msg)
        
        # Update activations based on signal strength
        signal_strength = h_new.norm(dim=-1)
        alpha_new = self.soft_threshold(alpha + signal_strength * 0.1, self.thresholds)
        
        # Residual connection with layer norm
        h_out = self.layer_norm(h_new + h)
        h_out = h_out * alpha_new.unsqueeze(-1)
        
        return h_out, alpha_new

    def local_inhibition(self, alpha):
        """Apply local inhibition based on spatial proximity."""
        pos = self.get_positions().detach()
        dist = torch.cdist(pos, pos)
        neighborhood = (dist < self.inhibition_radius).float()
        neighbor_sum = torch.einsum('ij,bj->bi', neighborhood, alpha) + 1e-6
        alpha_normalized = alpha * neighborhood.sum(dim=1) / neighbor_sum
        return alpha_normalized.clamp(0, 1)

    def forward(self, x):
        """Forward pass through RSGN."""
        batch_size = x.shape[0]
        
        # Compute connection weights
        W = self.compute_connection_weights()
        
        # Initial ignition
        alpha = self.ignite(x)
        
        # Initialize hidden states
        x_pooled = x.mean(dim=1)
        h = self.init_state(x_pooled).unsqueeze(1)
        h = h.expand(-1, self.num_nodes, -1).clone()
        h = h * alpha.unsqueeze(-1)

        # Resonant dynamics
        for step in range(self.num_steps):
            h, alpha = self.propagate(h, alpha, W)
            alpha = self.local_inhibition(alpha)

        # Output: weighted sum of active node states
        output = (h * alpha.unsqueeze(-1)).sum(dim=1)
        
        return output, alpha


class RSGClassifier(nn.Module):
    """RSGN with classification head."""
    
    def __init__(self, num_classes, num_nodes=256, dim_input=64, dim_hidden=128, **kwargs):
        super().__init__()
        self.rsg = RSGNetwork(num_nodes=num_nodes, dim_input=dim_input,
                              dim_hidden=dim_hidden, **kwargs)
        self.classifier = nn.Linear(dim_hidden, num_classes)

    def forward(self, x):
        features, alpha = self.rsg(x)
        logits = self.classifier(features)
        return logits, alpha


print("RSGN model defined.")

In [ ]:
#@title Cell 4: Hebbian Learning Module {display-mode: "form"}

class HebbianLearner:
    """Two-timescale Hebbian learning for structural plasticity.
    
    Implements slow structural learning alongside fast gradient descent:
    - Updates affinity factors based on co-activation patterns
    - Adapts thresholds to maintain target sparsity
    - Supports synaptic pruning and sprouting
    """
    
    def __init__(self, model, lr_affinity=0.001, lr_position=0.0001,
                 lr_threshold=0.001, decay=0.995, sparsity_target=0.1,
                 history_size=100):
        self.model = model
        self.lr_affinity = lr_affinity
        self.lr_position = lr_position
        self.lr_threshold = lr_threshold
        self.decay = decay
        self.sparsity_target = sparsity_target
        self.activation_history = deque(maxlen=history_size)
        self.stats = {'hebbian_updates': 0}

    def update(self, alpha, reward=1.0):
        """Apply Hebbian update based on activation patterns."""
        with torch.no_grad():
            device = alpha.device
            alpha_mean = alpha.mean(dim=0)
            self.activation_history.append(alpha_mean.cpu().clone())

            # Hebbian affinity update: neurons that fire together, wire together
            outer = alpha_mean.unsqueeze(1) * alpha_mean.unsqueeze(0)
            delta_affinity = self.lr_affinity * outer * reward
            self._update_affinity_factors(delta_affinity, device)

            # Threshold adaptation to maintain sparsity
            sparsity_error = alpha_mean - self.sparsity_target
            self.model.thresholds.data += self.lr_threshold * sparsity_error
            self.model.thresholds.data.clamp_(min=0.01, max=0.99)

            self.stats['hebbian_updates'] += 1

    def _update_affinity_factors(self, delta_affinity, device):
        """Update low-rank affinity factors."""
        u = self.model.affinity_u.data
        v = self.model.affinity_v.data
        
        current_pre_sigmoid = u @ v.T
        target = torch.sigmoid(current_pre_sigmoid) + delta_affinity.to(device)
        target = target * self.decay + torch.sigmoid(current_pre_sigmoid) * (1 - self.decay)
        
        error = torch.sigmoid(current_pre_sigmoid) - target
        sigmoid_deriv = torch.sigmoid(current_pre_sigmoid) * (1 - torch.sigmoid(current_pre_sigmoid))
        grad_u = (error * sigmoid_deriv) @ v
        
        self.model.affinity_u.data -= self.lr_affinity * grad_u

    def prune_and_sprout(self, threshold_low=0.01, threshold_high=0.9):
        """Prune weak connections and sprout new ones."""
        if len(self.activation_history) < 10:
            return
            
        with torch.no_grad():
            # Compute average activation
            avg_activation = torch.stack(list(self.activation_history)).mean(dim=0)
            
            # Identify underused nodes
            underused = (avg_activation < threshold_low).to(self.model.positions.device)
            
            # Slightly randomize positions of underused nodes
            if underused.any():
                noise = torch.randn_like(self.model.positions) * 0.05
                self.model.positions.data[underused] += noise[underused]
                self.model.positions.data = project_to_poincare(
                    self.model.positions.data, max_norm=0.95
                )


print("Hebbian learning module defined.")

In [ ]:
#@title Cell 5: Baseline Models {display-mode: "form"}

class MLPClassifier(nn.Module):
    """Multi-layer perceptron baseline."""
    
    def __init__(self, seq_len, dim_input, dim_hidden, num_classes):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(seq_len * dim_input, dim_hidden),
            nn.ReLU(),
            nn.Linear(dim_hidden, dim_hidden),
            nn.ReLU(),
            nn.Linear(dim_hidden, num_classes)
        )

    def forward(self, x):
        return self.net(self.flatten(x)), None


class TransformerClassifier(nn.Module):
    """Standard Transformer baseline."""
    
    def __init__(self, dim_input, dim_hidden, num_classes, num_heads=4, num_layers=2):
        super().__init__()
        self.embed = nn.Linear(dim_input, dim_hidden)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim_hidden, nhead=num_heads,
            dim_feedforward=dim_hidden * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(dim_hidden, num_classes)

    def forward(self, x):
        x = self.embed(x)
        x = self.transformer(x)
        x = x.mean(dim=1)  # Global average pooling
        return self.classifier(x), None


class SparseTransformerClassifier(nn.Module):
    """Sparse Transformer with local + strided attention."""
    
    def __init__(self, dim_input, dim_hidden, num_classes, num_heads=4, num_layers=2):
        super().__init__()
        self.embed = nn.Linear(dim_input, dim_hidden)
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(dim_hidden, num_heads, batch_first=True)
            for _ in range(num_layers)
        ])
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim_hidden, dim_hidden * 4),
                nn.GELU(),
                nn.Linear(dim_hidden * 4, dim_hidden)
            ) for _ in range(num_layers)
        ])
        self.norms1 = nn.ModuleList([nn.LayerNorm(dim_hidden) for _ in range(num_layers)])
        self.norms2 = nn.ModuleList([nn.LayerNorm(dim_hidden) for _ in range(num_layers)])
        self.classifier = nn.Linear(dim_hidden, num_classes)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        x = self.embed(x)
        
        # Create sparse attention mask (local window + strided)
        mask = torch.ones(seq_len, seq_len, device=x.device)
        for i in range(seq_len):
            # Local window of size 5
            start, end = max(0, i - 2), min(seq_len, i + 3)
            mask[i, start:end] = 0
            # Strided attention every 4 positions
            for j in range(0, seq_len, 4):
                mask[i, j] = 0
        mask = mask.bool()
        
        for attn, ffn, norm1, norm2 in zip(self.attention_layers, self.ffn_layers, 
                                           self.norms1, self.norms2):
            attn_out, _ = attn(x, x, x, attn_mask=mask)
            x = norm1(x + attn_out)
            x = norm2(x + ffn(x))
            
        return self.classifier(x.mean(dim=1)), None


class LSTMClassifier(nn.Module):
    """Bidirectional LSTM baseline."""
    
    def __init__(self, dim_input, dim_hidden, num_classes, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(dim_input, dim_hidden, num_layers, 
                           batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(dim_hidden * 2, num_classes)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=-1)  # Concat forward and backward
        return self.classifier(h), None


def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Baseline models defined: MLP, Transformer, Sparse Transformer, LSTM")

In [ ]:
#@title Cell 6: Data Generation {display-mode: "form"}

class SequenceDataset(Dataset):
    """Simple dataset wrapper."""
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_dataloaders(X, y, train_ratio=0.8, batch_size=64, seed=42):
    """Create train/val dataloaders."""
    np.random.seed(seed)
    n = len(y)
    indices = np.random.permutation(n)
    train_size = int(n * train_ratio)
    train_idx, val_idx = indices[:train_size], indices[train_size:]
    
    train_dataset = SequenceDataset(X[train_idx], y[train_idx])
    val_dataset = SequenceDataset(X[val_idx], y[val_idx])
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader


def generate_hierarchical_data(num_samples=5000, seq_len=64, dim=32, 
                               num_classes=20, noise=0.3, seed=42):
    """Generate challenging hierarchical classification data.
    
    Creates multi-scale patterns that require hierarchical processing:
    - Level 1: Local patterns (5-step subsequences)
    - Level 2: Mid-range patterns (quarter-sequence markers)
    - Level 3: Global patterns (entire sequence bias)
    """
    np.random.seed(seed)
    
    # Base noise
    X = np.random.randn(num_samples, seq_len, dim) * noise
    y = np.random.randint(0, num_classes, num_samples)

    # Generate class-specific patterns at multiple scales
    level1_patterns = np.random.randn(num_classes, 8, 5, dim) * 0.15  # 8 local patterns per class
    level2_patterns = np.random.randn(num_classes, 4, dim) * 0.10    # 4 mid-range markers
    level3_patterns = np.random.randn(num_classes, dim) * 0.08      # 1 global pattern

    for i in range(num_samples):
        c = y[i]
        
        # Level 1: Insert random local patterns
        num_local = np.random.randint(2, 5)
        for _ in range(num_local):
            pos = np.random.randint(0, max(1, seq_len - 5))
            pattern_idx = np.random.randint(0, 8)
            end_pos = min(pos + 5, seq_len)
            X[i, pos:end_pos] += level1_patterns[c, pattern_idx, :end_pos-pos]
        
        # Level 2: Insert quarter-position markers
        for p in range(4):
            pos = (seq_len // 4) * p + np.random.randint(0, max(1, seq_len // 8))
            pos = min(pos, seq_len - 1)
            X[i, pos] += level2_patterns[c, p]
        
        # Level 3: Global bias
        X[i] += level3_patterns[c] * 0.05

    return torch.FloatTensor(X), torch.LongTensor(y)


def generate_long_range_data(num_samples=3000, seq_len=128, dim=32, 
                             num_classes=10, seed=42):
    """Generate data requiring long-range dependency learning.
    
    Class is determined by patterns at the BEGINNING and END of sequence.
    Models must attend across the full sequence to classify correctly.
    """
    np.random.seed(seed)
    
    # Base noise
    X = np.random.randn(num_samples, seq_len, dim) * 0.3
    y = np.zeros(num_samples, dtype=np.int64)

    # Beginning and end patterns per class
    begin_patterns = np.random.randn(num_classes, 8, dim) * 0.5
    end_patterns = np.random.randn(num_classes, 8, dim) * 0.5

    for i in range(num_samples):
        c = np.random.randint(0, num_classes)
        y[i] = c
        
        # Insert class patterns at beginning and end
        X[i, :8] += begin_patterns[c]
        X[i, -8:] += end_patterns[c]

    return torch.FloatTensor(X), torch.LongTensor(y)


print("Data generation functions defined.")

In [ ]:
#@title Cell 7: Training Function {display-mode: "form"}

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, 
                hebbian=None, model_name="Model", verbose=True):
    """Train a model and return history."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'sparsity': []}
    best_val_acc = 0

    for epoch in range(epochs):
        # Training
        model.train()
        total_correct, total_samples = 0, 0
        total_loss = 0
        total_sparsity, n_batches = 0, 0

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            logits, alpha = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # Hebbian update (if enabled)
            if hebbian is not None and alpha is not None:
                hebbian.update(alpha.detach(), reward=-loss.item())

            # Track metrics
            preds = logits.argmax(dim=-1)
            total_correct += (preds == batch_y).sum().item()
            total_samples += batch_y.size(0)
            total_loss += loss.item()
            
            if alpha is not None:
                total_sparsity += (alpha > 0.01).float().mean().item()
            n_batches += 1

        scheduler.step()
        
        train_acc = total_correct / total_samples
        train_loss = total_loss / n_batches
        sparsity = total_sparsity / n_batches if n_batches > 0 else 0
        
        history['train_acc'].append(train_acc)
        history['train_loss'].append(train_loss)
        history['sparsity'].append(sparsity)

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits, _ = model(batch_x)
                preds = logits.argmax(dim=-1)
                val_correct += (preds == batch_y).sum().item()
                val_total += batch_y.size(0)

        val_acc = val_correct / val_total
        history['val_acc'].append(val_acc)
        best_val_acc = max(best_val_acc, val_acc)

        # Hebbian pruning/sprouting
        if hebbian is not None and (epoch + 1) % 10 == 0:
            hebbian.prune_and_sprout()

        # Progress output
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}: "
                  f"Train={train_acc*100:.1f}%, Val={val_acc*100:.1f}%, "
                  f"Loss={train_loss:.3f}")

    return history, best_val_acc


print("Training function defined.")

In [ ]:
#@title Cell 8: EXPERIMENT 1 - Hierarchical Classification {display-mode: "form"}

print("="*70)
print("EXPERIMENT 1: Hierarchical Classification Task")
print("="*70)
print("Settings: 20 classes, 64 sequence length, 0.3 noise, 50 epochs")
print("Models: RSGN+Hebbian, RSGN, Transformer, Sparse Trans., LSTM, MLP")
print("Runs per model: 3 (for statistical significance)")
print("="*70)

set_seed(42)

# Experiment settings
SEQ_LEN = 64
DIM_INPUT = 32
DIM_HIDDEN = 128
NUM_CLASSES = 20
NUM_SAMPLES = 5000
EPOCHS = 50
N_RUNS = 3

# Generate data
print("\nGenerating hierarchical classification data...")
X_hier, y_hier = generate_hierarchical_data(
    num_samples=NUM_SAMPLES, seq_len=SEQ_LEN,
    dim=DIM_INPUT, num_classes=NUM_CLASSES, noise=0.3
)
print(f"Data shape: {X_hier.shape}, Classes: {NUM_CLASSES}")

# Model configurations
model_configs = [
    ('RSGN+Hebbian', 
     lambda: RSGClassifier(num_classes=NUM_CLASSES, num_nodes=256,
                           dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN, num_steps=5),
     True),
    ('RSGN',
     lambda: RSGClassifier(num_classes=NUM_CLASSES, num_nodes=256,
                           dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN, num_steps=5),
     False),
    ('Transformer',
     lambda: TransformerClassifier(dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN,
                                   num_classes=NUM_CLASSES, num_heads=4, num_layers=2),
     False),
    ('Sparse Trans.',
     lambda: SparseTransformerClassifier(dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN,
                                         num_classes=NUM_CLASSES, num_heads=4, num_layers=2),
     False),
    ('LSTM',
     lambda: LSTMClassifier(dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN,
                           num_classes=NUM_CLASSES, num_layers=2),
     False),
    ('MLP',
     lambda: MLPClassifier(seq_len=SEQ_LEN, dim_input=DIM_INPUT,
                          dim_hidden=DIM_HIDDEN, num_classes=NUM_CLASSES),
     False),
]

# Run experiments
hier_results = {}
hier_histories = {}

start_time = time.time()

for model_name, model_fn, use_hebbian in model_configs:
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")
    
    accuracies = []
    params = 0
    
    for run in range(N_RUNS):
        print(f"\n  Run {run+1}/{N_RUNS}:")
        set_seed(42 + run)
        
        train_loader, val_loader = create_dataloaders(X_hier, y_hier, batch_size=64)
        model = model_fn()
        params = count_parameters(model)
        
        hebbian = None
        if use_hebbian:
            hebbian = HebbianLearner(model.rsg, lr_affinity=0.002)
        
        history, best_acc = train_model(
            model, train_loader, val_loader,
            epochs=EPOCHS, hebbian=hebbian,
            model_name=model_name, verbose=True
        )
        
        accuracies.append(best_acc)
        
        if run == 0:
            hier_histories[model_name] = history
    
    # Store results
    acc_mean = np.mean(accuracies)
    acc_std = np.std(accuracies)
    hier_results[model_name] = {
        'accuracy_mean': float(acc_mean),
        'accuracy_std': float(acc_std),
        'parameters': params,
        'all_accuracies': [float(a) for a in accuracies]
    }
    
    print(f"\n>>> {model_name}: {acc_mean*100:.1f}% +/- {acc_std*100:.1f}% ({params:,} params)")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"Experiment 1 completed in {elapsed/60:.1f} minutes")
print(f"{'='*70}")

# Save results
with open('results/hierarchical_results.json', 'w') as f:
    json.dump(hier_results, f, indent=2)

print("\nResults saved to: results/hierarchical_results.json")

In [ ]:
#@title Cell 9: EXPERIMENT 2 - Long-Range Dependencies {display-mode: "form"}

print("\n" + "="*70)
print("EXPERIMENT 2: Long-Range Dependency Task")
print("="*70)
print("Settings: 10 classes, 128 sequence length, 50 epochs")
print("Task: Classify based on patterns at sequence START and END")
print("="*70)

set_seed(42)

SEQ_LEN_LR = 128
NUM_CLASSES_LR = 10
NUM_SAMPLES_LR = 3000

# Generate data
print("\nGenerating long-range dependency data...")
X_lr, y_lr = generate_long_range_data(
    num_samples=NUM_SAMPLES_LR, seq_len=SEQ_LEN_LR,
    dim=DIM_INPUT, num_classes=NUM_CLASSES_LR
)
print(f"Data shape: {X_lr.shape}, Sequence length: {SEQ_LEN_LR}")

# Model configurations (subset for long-range task)
lr_configs = [
    ('RSGN+Hebbian',
     lambda: RSGClassifier(num_classes=NUM_CLASSES_LR, num_nodes=256,
                           dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN, num_steps=7),
     True),
    ('RSGN',
     lambda: RSGClassifier(num_classes=NUM_CLASSES_LR, num_nodes=256,
                           dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN, num_steps=7),
     False),
    ('Transformer',
     lambda: TransformerClassifier(dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN,
                                   num_classes=NUM_CLASSES_LR, num_heads=4, num_layers=3),
     False),
    ('LSTM',
     lambda: LSTMClassifier(dim_input=DIM_INPUT, dim_hidden=DIM_HIDDEN,
                           num_classes=NUM_CLASSES_LR, num_layers=2),
     False),
]

# Run experiments
lr_results = {}
lr_histories = {}

start_time = time.time()

for model_name, model_fn, use_hebbian in lr_configs:
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")
    
    accuracies = []
    params = 0
    
    for run in range(N_RUNS):
        print(f"\n  Run {run+1}/{N_RUNS}:")
        set_seed(42 + run)
        
        train_loader_lr, val_loader_lr = create_dataloaders(X_lr, y_lr, batch_size=32)
        model = model_fn()
        params = count_parameters(model)
        
        hebbian = None
        if use_hebbian:
            hebbian = HebbianLearner(model.rsg, lr_affinity=0.002)
        
        history, best_acc = train_model(
            model, train_loader_lr, val_loader_lr,
            epochs=EPOCHS, hebbian=hebbian,
            model_name=model_name, verbose=True
        )
        
        accuracies.append(best_acc)
        
        if run == 0:
            lr_histories[model_name] = history
    
    acc_mean = np.mean(accuracies)
    acc_std = np.std(accuracies)
    lr_results[model_name] = {
        'accuracy_mean': float(acc_mean),
        'accuracy_std': float(acc_std),
        'parameters': params,
        'all_accuracies': [float(a) for a in accuracies]
    }
    
    print(f"\n>>> {model_name}: {acc_mean*100:.1f}% +/- {acc_std*100:.1f}%")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"Experiment 2 completed in {elapsed/60:.1f} minutes")
print(f"{'='*70}")

# Save results
with open('results/long_range_results.json', 'w') as f:
    json.dump(lr_results, f, indent=2)

print("\nResults saved to: results/long_range_results.json")

In [ ]:
#@title Cell 10: EXPERIMENT 3 - Ablation Study {display-mode: "form"}

print("\n" + "="*70)
print("EXPERIMENT 3: Ablation Study")
print("="*70)
print("Analyzing contribution of each RSGN component")
print("="*70)

set_seed(42)

# Use hierarchical data
train_loader, val_loader = create_dataloaders(X_hier, y_hier, batch_size=64)

ablation_configs = [
    ('Full RSGN (256 nodes, 5 steps, Hebbian)', {'num_steps': 5, 'num_nodes': 256}, True),
    ('No Hebbian', {'num_steps': 5, 'num_nodes': 256}, False),
    ('3 Steps', {'num_steps': 3, 'num_nodes': 256}, True),
    ('1 Step', {'num_steps': 1, 'num_nodes': 256}, True),
    ('128 Nodes', {'num_steps': 5, 'num_nodes': 128}, True),
    ('512 Nodes', {'num_steps': 5, 'num_nodes': 512}, True),
]

ablation_results = {}

start_time = time.time()

for config_name, config, use_hebbian in ablation_configs:
    print(f"\n{'='*50}")
    print(f"Configuration: {config_name}")
    print(f"{'='*50}")
    
    set_seed(42)
    
    model = RSGClassifier(
        num_classes=NUM_CLASSES,
        num_nodes=config['num_nodes'],
        dim_input=DIM_INPUT,
        dim_hidden=DIM_HIDDEN,
        num_steps=config['num_steps']
    )
    
    hebbian = HebbianLearner(model.rsg, lr_affinity=0.002) if use_hebbian else None
    
    history, best_acc = train_model(
        model, train_loader, val_loader,
        epochs=EPOCHS, hebbian=hebbian,
        model_name=config_name, verbose=True
    )
    
    final_sparsity = history['sparsity'][-1] if history['sparsity'] else 0
    
    ablation_results[config_name] = {
        'accuracy_mean': float(best_acc),
        'accuracy_std': 0.02,  # Single run, estimate
        'sparsity_mean': float(final_sparsity),
        'parameters': count_parameters(model)
    }
    
    print(f"\n>>> {config_name}: {best_acc*100:.1f}%, sparsity: {final_sparsity:.3f}")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"Experiment 3 completed in {elapsed/60:.1f} minutes")
print(f"{'='*70}")

# Save results
with open('results/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print("\nResults saved to: results/ablation_results.json")

In [ ]:
#@title Cell 11: Generate Figures {display-mode: "form"}

print("\n" + "="*70)
print("GENERATING PUBLICATION FIGURES")
print("="*70)

# Color palette
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3']

# Set publication style
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# FIGURE 1: Main Comparison Bar Chart
print("\nGenerating Figure 1: Main Comparison...")
fig, ax = plt.subplots(figsize=(10, 5))
models = list(hier_results.keys())
accs = [hier_results[m]['accuracy_mean'] * 100 for m in models]
stds = [hier_results[m]['accuracy_std'] * 100 for m in models]

bars = ax.bar(range(len(models)), accs, yerr=stds, capsize=5, 
              color=colors[:len(models)], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Hierarchical Classification Task (20 Classes, Noise 0.3)')

for bar, acc, std in zip(bars, accs, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/main_comparison.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/main_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/main_comparison.pdf")

# FIGURE 2: Long-Range Dependencies
print("\nGenerating Figure 2: Long-Range Dependencies...")
fig, ax = plt.subplots(figsize=(8, 5))
models_lr = list(lr_results.keys())
accs_lr = [lr_results[m]['accuracy_mean'] * 100 for m in models_lr]
stds_lr = [lr_results[m]['accuracy_std'] * 100 for m in models_lr]

bars = ax.bar(range(len(models_lr)), accs_lr, yerr=stds_lr, capsize=5,
              color=colors[:len(models_lr)], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(models_lr)))
ax.set_xticklabels(models_lr, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Long-Range Dependency Task (Sequence Length 128)')

for bar, acc, std in zip(bars, accs_lr, stds_lr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/long_range.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/long_range.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/long_range.pdf")

# FIGURE 3: Ablation Study
print("\nGenerating Figure 3: Ablation Study...")
fig, ax = plt.subplots(figsize=(10, 5))
configs = list(ablation_results.keys())
accs_abl = [ablation_results[c]['accuracy_mean'] * 100 for c in configs]

bars = ax.bar(range(len(configs)), accs_abl, color=colors[:len(configs)],
              edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(configs)))
ax.set_xticklabels(configs, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation Study: Component Contributions')

for bar, acc in zip(bars, accs_abl):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/ablation_study.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/ablation_study.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/ablation_study.pdf")

# FIGURE 4: Parameter Efficiency
print("\nGenerating Figure 4: Parameter Efficiency...")
fig, ax = plt.subplots(figsize=(9, 6))
for i, model in enumerate(models):
    params = hier_results[model]['parameters'] / 1000
    acc = hier_results[model]['accuracy_mean'] * 100
    ax.scatter(params, acc, s=150, color=colors[i], label=model, 
               edgecolor='black', linewidth=0.5, zorder=5)
    ax.annotate(model, (params, acc), textcoords="offset points",
               xytext=(5, 5), fontsize=9)

ax.set_xlabel('Parameters (K)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Parameter Efficiency: Accuracy vs Model Size')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('figures/parameter_efficiency.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/parameter_efficiency.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/parameter_efficiency.pdf")

# FIGURE 5: Combined Results (Multi-panel)
print("\nGenerating Figure 5: Combined Results...")
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A
ax = axes[0]
bars = ax.bar(range(len(models)), accs, yerr=stds, capsize=4, 
              color=colors[:len(models)], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(models)))
ax.set_xticklabels([m.replace(' ', '\n') for m in models], rotation=0, fontsize=8)
ax.set_ylabel('Accuracy (%)')
ax.set_title('A. Hierarchical Classification', fontweight='bold')

# Panel B
ax = axes[1]
bars = ax.bar(range(len(models_lr)), accs_lr, yerr=stds_lr, capsize=4,
              color=colors[:len(models_lr)], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(models_lr)))
ax.set_xticklabels([m.replace(' ', '\n') for m in models_lr], rotation=0, fontsize=8)
ax.set_ylabel('Accuracy (%)')
ax.set_title('B. Long-Range Dependencies', fontweight='bold')

# Panel C
ax = axes[2]
for i, model in enumerate(models):
    params = hier_results[model]['parameters'] / 1000
    acc = hier_results[model]['accuracy_mean'] * 100
    ax.scatter(params, acc, s=120, color=colors[i], label=model,
               edgecolor='black', linewidth=0.5, zorder=5)
ax.set_xlabel('Parameters (K)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('C. Parameter Efficiency', fontweight='bold')
ax.legend(loc='lower right', fontsize=7)

plt.tight_layout()
plt.savefig('figures/combined_results.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/combined_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/combined_results.pdf")

# FIGURE 6: Training Curves
print("\nGenerating Figure 6: Training Curves...")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (name, hist) in enumerate(hier_histories.items()):
    axes[0].plot(hist['train_acc'], label=name, color=colors[idx], alpha=0.8)
    axes[1].plot(hist['val_acc'], label=name, color=colors[idx], alpha=0.8)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Accuracy')
axes[0].set_title('Training Curves'); axes[0].legend(loc='lower right', fontsize=8)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Validation Curves'); axes[1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('figures/training_curves.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/training_curves.pdf")

# FIGURE 7: Sparsity Analysis
print("\nGenerating Figure 7: Sparsity Analysis...")
fig, ax = plt.subplots(figsize=(8, 5))
sparsities = [ablation_results[c]['sparsity_mean'] * 100 for c in configs]
accs_sp = [ablation_results[c]['accuracy_mean'] * 100 for c in configs]

ax.scatter(sparsities, accs_sp, s=150, color=colors[0], edgecolor='black', linewidth=1)
for i, cfg in enumerate(configs):
    ax.annotate(cfg, (sparsities[i], accs_sp[i]), textcoords="offset points",
               xytext=(5, 5), fontsize=8)

ax.set_xlabel('Active Node Fraction (%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Sparsity vs Accuracy Trade-off')

plt.tight_layout()
plt.savefig('figures/sparsity_analysis.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figures/sparsity_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: figures/sparsity_analysis.pdf")

print("\n" + "="*70)
print("All figures generated successfully!")
print("="*70)

In [ ]:
#@title Cell 12: Generate LaTeX Tables {display-mode: "form"}

print("\n" + "="*70)
print("LaTeX TABLES FOR MANUSCRIPT")
print("="*70)

# Get reference params for relative size
rsgn_params = hier_results.get('RSGN', hier_results.get('RSGN+Hebbian', {})).get('parameters', 1)

# TABLE 1: Main Results
print("\n% Table 1: Hierarchical Classification Results")
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Performance comparison on hierarchical classification task (20 classes, sequence length 64, noise level 0.3). Results show mean $\\pm$ std over 3 runs.}")
print("\\begin{tabular}{lccc}")
print("\\toprule")
print("Model & Accuracy (\\%) & Parameters & Rel. Size \\\\")
print("\\midrule")

sorted_hier = sorted(hier_results.items(), key=lambda x: x[1]['accuracy_mean'], reverse=True)
for model_name, res in sorted_hier:
    acc = f"{res['accuracy_mean']*100:.1f} $\\pm$ {res['accuracy_std']*100:.1f}"
    params = f"{res['parameters']:,}"
    rel_size = f"{res['parameters']/rsgn_params:.1f}$\\times$"
    print(f"{model_name} & {acc} & {params} & {rel_size} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:main_results}")
print("\\end{table}")

# TABLE 2: Long-Range Results
print("\n% Table 2: Long-Range Dependency Results")
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Performance on long-range dependency task (sequence length 128). Class determined by patterns at sequence start and end.}")
print("\\begin{tabular}{lcc}")
print("\\toprule")
print("Model & Accuracy (\\%) & Parameters \\\\")
print("\\midrule")

sorted_lr = sorted(lr_results.items(), key=lambda x: x[1]['accuracy_mean'], reverse=True)
for model_name, res in sorted_lr:
    acc = f"{res['accuracy_mean']*100:.1f} $\\pm$ {res['accuracy_std']*100:.1f}"
    params = f"{res['parameters']:,}"
    print(f"{model_name} & {acc} & {params} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:long_range}")
print("\\end{table}")

# TABLE 3: Ablation Study
print("\n% Table 3: Ablation Study")
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Ablation study of RSGN components on hierarchical classification task.}")
print("\\begin{tabular}{lcc}")
print("\\toprule")
print("Configuration & Accuracy (\\%) & Sparsity \\\\")
print("\\midrule")

sorted_abl = sorted(ablation_results.items(), key=lambda x: x[1]['accuracy_mean'], reverse=True)
for config_name, res in sorted_abl:
    print(f"{config_name} & {res['accuracy_mean']*100:.1f} & {res['sparsity_mean']:.2f} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:ablation}")
print("\\end{table}")

# Save to file
latex_tables = []
latex_tables.append("% Auto-generated LaTeX tables from RSGN experiments\n")
latex_tables.append("% Copy these directly into your manuscript\n\n")

# Add all tables to file content
with open('results/latex_tables.tex', 'w') as f:
    f.write("% Auto-generated LaTeX tables from RSGN experiments\n")
    f.write("% Copy these directly into your manuscript\n\n")
    
    # Table 1
    f.write("% Table 1: Hierarchical Classification\n")
    f.write("\\begin{table}[h]\n")
    f.write("\\centering\n")
    f.write("\\caption{Performance on hierarchical classification (20 classes, seq 64, noise 0.3).}\n")
    f.write("\\begin{tabular}{lccc}\n")
    f.write("\\toprule\n")
    f.write("Model & Accuracy (\\%) & Parameters & Rel. Size \\\\\n")
    f.write("\\midrule\n")
    for model_name, res in sorted_hier:
        acc = f"{res['accuracy_mean']*100:.1f} $\\pm$ {res['accuracy_std']*100:.1f}"
        params = f"{res['parameters']:,}"
        rel_size = f"{res['parameters']/rsgn_params:.1f}$\\times$"
        f.write(f"{model_name} & {acc} & {params} & {rel_size} \\\\\n")
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\label{tab:main_results}\n")
    f.write("\\end{table}\n\n")
    
    # Table 2
    f.write("% Table 2: Long-Range Dependencies\n")
    f.write("\\begin{table}[h]\n")
    f.write("\\centering\n")
    f.write("\\caption{Performance on long-range dependency task (seq length 128).}\n")
    f.write("\\begin{tabular}{lcc}\n")
    f.write("\\toprule\n")
    f.write("Model & Accuracy (\\%) & Parameters \\\\\n")
    f.write("\\midrule\n")
    for model_name, res in sorted_lr:
        acc = f"{res['accuracy_mean']*100:.1f} $\\pm$ {res['accuracy_std']*100:.1f}"
        params = f"{res['parameters']:,}"
        f.write(f"{model_name} & {acc} & {params} \\\\\n")
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\label{tab:long_range}\n")
    f.write("\\end{table}\n\n")
    
    # Table 3
    f.write("% Table 3: Ablation Study\n")
    f.write("\\begin{table}[h]\n")
    f.write("\\centering\n")
    f.write("\\caption{Ablation study of RSGN components.}\n")
    f.write("\\begin{tabular}{lcc}\n")
    f.write("\\toprule\n")
    f.write("Configuration & Accuracy (\\%) & Sparsity \\\\\n")
    f.write("\\midrule\n")
    for config_name, res in sorted_abl:
        f.write(f"{config_name} & {res['accuracy_mean']*100:.1f} & {res['sparsity_mean']:.2f} \\\\\n")
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\label{tab:ablation}\n")
    f.write("\\end{table}\n")

print("\n" + "="*70)
print("LaTeX tables saved to: results/latex_tables.tex")
print("="*70)

In [ ]:
#@title Cell 13: Final Summary {display-mode: "form"}

print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)

print("\n### HIERARCHICAL CLASSIFICATION (20 classes) ###")
for model_name, res in sorted_hier:
    print(f"  {model_name:20s}: {res['accuracy_mean']*100:5.1f}% +/- {res['accuracy_std']*100:.1f}% ({res['parameters']:,} params)")

print("\n### LONG-RANGE DEPENDENCIES (seq 128) ###")
for model_name, res in sorted_lr:
    print(f"  {model_name:20s}: {res['accuracy_mean']*100:5.1f}% +/- {res['accuracy_std']*100:.1f}%")

print("\n### ABLATION STUDY ###")
for config_name, res in sorted_abl:
    print(f"  {config_name:45s}: {res['accuracy_mean']*100:5.1f}%")

print("\n### KEY FINDINGS ###")
rsgn_hier = hier_results.get('RSGN+Hebbian', hier_results.get('RSGN', {}))
trans_hier = hier_results.get('Transformer', {})
if rsgn_hier and trans_hier:
    param_ratio = trans_hier['parameters'] / rsgn_hier['parameters']
    print(f"  - RSGN uses {param_ratio:.1f}x fewer parameters than Transformer")

rsgn_lr = lr_results.get('RSGN+Hebbian', lr_results.get('RSGN', {}))
trans_lr = lr_results.get('Transformer', {})
if rsgn_lr and trans_lr:
    diff = (rsgn_lr['accuracy_mean'] - trans_lr['accuracy_mean']) * 100
    print(f"  - On long-range task: RSGN outperforms Transformer by {diff:+.1f}%")
    print(f"  - RSGN achieves {rsgn_lr['accuracy_mean']*100:.1f}% vs Transformer {trans_lr['accuracy_mean']*100:.1f}%")

print("\n### FILES GENERATED ###")
print("  Results:")
print("    - results/hierarchical_results.json")
print("    - results/long_range_results.json")
print("    - results/ablation_results.json")
print("    - results/latex_tables.tex")
print("  Figures:")
print("    - figures/main_comparison.pdf")
print("    - figures/long_range.pdf")
print("    - figures/ablation_study.pdf")
print("    - figures/parameter_efficiency.pdf")
print("    - figures/combined_results.pdf")
print("    - figures/training_curves.pdf")
print("    - figures/sparsity_analysis.pdf")

print("\n" + "="*70)
print("ALL EXPERIMENTS COMPLETE!")
print("="*70)

In [ ]:
#@title Cell 14: Download Results {display-mode: "form"}

import shutil
from google.colab import files

print("Creating zip archives...")

# Create zip files
shutil.make_archive('RSGN_results', 'zip', '.', 'results')
shutil.make_archive('RSGN_figures', 'zip', '.', 'figures')

print("\nDownloading...")
print("(Check your browser's download folder)")

files.download('RSGN_results.zip')
files.download('RSGN_figures.zip')

print("\nDownload complete!")
print("\nExtract the zip files to get:")
print("  - RSGN_results.zip → JSON files and LaTeX tables")
print("  - RSGN_figures.zip → PDF and PNG figures for manuscript")